# Regular-Time Production Scheduling Model

This notebook converts a product forecast into a daily production plan using only staffed regular shifts. Overtime is intentionally excluded. The `Unscheduled` quantity in the forecast-attainment report is the handoff for a separate overtime solver.

The planning horizon may be a month, two weeks, one week, or any other dated period supplied by `Work_Total`.

## Terminology used everywhere

- **Process**: a production stage or method, such as `Pri`, `Cut`, `Man`, `Sew`, `QaC`, or `Pac`.
- **Line**: one physical production resource inside a process, such as `Pri1`, `Pri2`, or `Pri3`.
- **POL time**: person-time per unit for a `(product, process)` pair, measured in milliseconds.
- **Route**: the ordered list of processes required by a product.
- **WIP after a process**: units that completed that process but have not completed the next process.

| Model item | Correct index |
|---|---|
| POL time | product, process |
| Route flow and WIP | product, process, day |
| Available labor | line, shift, day |
| Regular schedule | product, line, shift, day |

A process may have several lines. Those lines share the process's POL standard but have separate workers, shifts, availability, production decisions, and capacity constraints.


In [ ]:
from pathlib import Path
import importlib
import math
import re

from IPython.display import display
import pandas as pd
import pulp as pl
from openpyxl import load_workbook

from utils import util_display as display_utils
from utils import util_export_schedule_2xl as export_utils

# Reload local helper modules while developing in Jupyter.
importlib.reload(display_utils)
importlib.reload(export_utils)


## 1. Settings and file paths

Model switches and constants live in one cell so the behavior can be changed without searching through the constraints.


In [ ]:
# ---------- Production time units ----------
MS_PER_HOUR = 3_600_000

# ---------- Time-utilization policy ----------
MIN_TIME_UTILIZATION = 0.70

# ---------- Objective and route policy ----------
ALLOW_SAME_DAY_TRANSFER = True
CLEAR_ENDING_WIP = False

# ---------- Solver settings (HiGHS expects seconds) ----------
SOLVER_RELATIVE_GAP = 0.01
SOLVER_TIME_LIMIT_SECONDS = 1800
SOLVER_LOG = True
SOLUTION_TOLERANCE = 1e-6

# ---------- Workbook layout ----------
LAST_MIP_SHEET = "Last_MIP"
LAST_MIP_REQUIRED = False  # False lets the first planning period start with zero WIP.
WRITE_MIP_SNAPSHOT = True
WORK_TOTAL_HEADER_ROW = 2
WORK_TOTAL_DATE_START_COLUMN = 9

# ---------- Files ----------
DATA_DIR = Path("xlconfigs")
PRODUCTION_INPUT = DATA_DIR / "production_input.xlsm"
LABOR_INPUT = DATA_DIR / "labor_input.xlsm"
PLANNING_INPUT = DATA_DIR / "planning_input.xlsm"

for input_file in (PRODUCTION_INPUT, LABOR_INPUT):
    if not input_file.exists():
        raise FileNotFoundError(f"Input file not found: {input_file.resolve()}")


## 2. Small validation helpers

These functions only validate or normalize input data. They do not build solver variables or constraints.


In [ ]:
def require_columns(frame, required_columns, sheet_name):
    """Raise a readable error when an Excel sheet is missing required columns."""
    missing_columns = [
        column
        for column in required_columns
        if column not in frame.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{sheet_name} is missing required columns: {missing_columns}"
        )


def clean_identifier(value):
    """Use one consistent string representation for product, line, and shift IDs."""
    return str(value).strip()


def clean_identifier_list(series):
    """Drop blank Excel rows and return clean identifiers in their original order."""
    nonblank_values = series.dropna()
    return [clean_identifier(value) for value in nonblank_values]


def require_unique(values, label):
    """Reject duplicate master-data identifiers instead of silently overwriting them."""
    value_series = pd.Series(values, dtype="object")
    duplicates = value_series[value_series.duplicated()].unique().tolist()

    if duplicates:
        raise ValueError(f"Duplicate {label}: {duplicates}")


def as_integers(series, label):
    """Convert an Excel quantity column to integer values with validation."""
    numeric = pd.to_numeric(series, errors="raise")

    if numeric.isna().any():
        raise ValueError(f"{label} contains blank values.")

    fractional = (numeric - numeric.round()).abs() > 1e-6
    if fractional.any():
        bad_values = numeric[fractional].head().tolist()
        raise ValueError(f"{label} must contain integers. Examples: {bad_values}")

    return numeric.round().astype("int64")


def as_nonnegative_integers(series, label):
    """Convert a nonnegative Excel quantity column to validated integers."""
    integers = as_integers(series, label)

    if (integers < 0).any():
        raise ValueError(f"{label} cannot contain negative values.")

    return integers


def process_from_line(line):
    """Map a physical line such as Pri12 to its process Pri."""
    return re.sub(r"\d+$", "", clean_identifier(line)).strip()


def safe_ratio(numerator, denominator):
    """Return a ratio without hiding a positive requirement behind zero capacity."""
    if denominator > 0:
        return numerator / denominator

    if numerator > 0:
        return float("inf")

    return 0.0


## 3. Load the production workbook

Required sheets:

- `Master_List`
- `Product_Forecast`
- `POL_Matrix`
- `Line_Config`
- `Product_Config`

`Last_MIP` is optional only when `LAST_MIP_REQUIRED = False`.


In [ ]:
PRODUCTION_SHEETS = [
    "Master_List",
    "Product_Forecast",
    "POL_Matrix",
    "Line_Config",
    "Product_Config",
]

with pd.ExcelFile(PRODUCTION_INPUT, engine="openpyxl") as workbook:
    missing_sheets = [
        sheet
        for sheet in PRODUCTION_SHEETS
        if sheet not in workbook.sheet_names
    ]

    if missing_sheets:
        raise ValueError(
            f"{PRODUCTION_INPUT.name} is missing sheets: {missing_sheets}"
        )

    production_sheets = {
        sheet: workbook.parse(sheet)
        for sheet in PRODUCTION_SHEETS
    }

    if LAST_MIP_SHEET in workbook.sheet_names:
        last_mip_raw = workbook.parse(LAST_MIP_SHEET)
    elif LAST_MIP_REQUIRED:
        raise ValueError(
            f"{PRODUCTION_INPUT.name} needs a {LAST_MIP_SHEET!r} sheet."
        )
    else:
        last_mip_raw = pd.DataFrame()

master_list = production_sheets["Master_List"].copy()
product_forecast = production_sheets["Product_Forecast"].copy()
pol_matrix_raw = production_sheets["POL_Matrix"].copy()
line_config = production_sheets["Line_Config"].copy()
product_config = production_sheets["Product_Config"].copy()  # Reserved for later use.


### 3.1 Master lists, forecast, and line-specific settings

`Line_Config` supplies only line identity and `Minimum_Input`. Unit-based `Max_KPI` and `Min_KPI` are no longer used because products can have different POL times.

The maximum time KPI now comes directly from each line-shift-day's available worker-time in `Work_Total`. The minimum time KPI is `MIN_TIME_UTILIZATION` (70%) of that same dated capacity, so rotating shifts and absences are handled automatically by the hours already calculated in Excel.

In [ ]:
# ---------- Master_List ----------
require_columns(
    master_list,
    ["Product_List", "Production_Lines"],
    "Master_List",
)

products = clean_identifier_list(master_list["Product_List"])
lines = clean_identifier_list(master_list["Production_Lines"])

require_unique(products, "products in Master_List")
require_unique(lines, "lines in Master_List")

line_process = {
    line: process_from_line(line)
    for line in lines
}


# ---------- Product_Forecast ----------
forecast_columns = [
    "Product_List",
    "Forecast_Amount",
    "Profit_Per_Product",
]
require_columns(product_forecast, forecast_columns, "Product_Forecast")

product_forecast = product_forecast.loc[
    product_forecast["Product_List"].notna(),
    forecast_columns,
].copy()

product_forecast["Product_List"] = product_forecast["Product_List"].map(
    clean_identifier
)
require_unique(product_forecast["Product_List"], "products in Product_Forecast")

unknown_forecast_products = sorted(
    set(product_forecast["Product_List"]) - set(products)
)
missing_forecast_products = sorted(
    set(products) - set(product_forecast["Product_List"])
)

if unknown_forecast_products or missing_forecast_products:
    raise ValueError(
        "Product_Forecast does not match Master_List. "
        f"Unknown={unknown_forecast_products}, missing={missing_forecast_products}"
    )

product_forecast = product_forecast.set_index("Product_List").reindex(products)
product_forecast["Forecast_Amount"] = as_nonnegative_integers(
    product_forecast["Forecast_Amount"],
    "Product_Forecast.Forecast_Amount",
)
product_forecast["Profit_Per_Product"] = pd.to_numeric(
    product_forecast["Profit_Per_Product"],
    errors="raise",
)

forecast = product_forecast["Forecast_Amount"].to_dict()
profit_per_product = product_forecast["Profit_Per_Product"].to_dict()


# ---------- Line_Config ----------
line_config_columns = [
    "Production_Line",
    "Minimum_Input",
]
require_columns(line_config, line_config_columns, "Line_Config")

line_config = line_config.loc[
    line_config["Production_Line"].notna(),
    line_config_columns,
].copy()

line_config["Production_Line"] = line_config["Production_Line"].map(
    clean_identifier
)
require_unique(line_config["Production_Line"], "lines in Line_Config")

unknown_config_lines = sorted(
    set(line_config["Production_Line"]) - set(lines)
)
missing_config_lines = sorted(
    set(lines) - set(line_config["Production_Line"])
)

if unknown_config_lines or missing_config_lines:
    raise ValueError(
        "Line_Config does not match Master_List. "
        f"Unknown={unknown_config_lines}, missing={missing_config_lines}"
    )

line_config = line_config.set_index("Production_Line").reindex(lines)
line_config["Minimum_Input"] = as_nonnegative_integers(
    line_config["Minimum_Input"],
    "Line_Config.Minimum_Input",
)

minimum_input_by_line = line_config["Minimum_Input"].to_dict()

### 3.2 Product routes and process-level POL times

The `POL_Matrix` columns are **processes**, and their order defines the route order. A zero means that the product skips that process.

POL is therefore stored by `(product, process)`. Physical lines enter later: each line uses the POL value of its parent process through `line_process[line]`.

Every positive POL value is person-time per unit. For example, if product `A` needs `300 ms` at process `Pri`, all `Pri` lines use that same `300 ms` standard. Their capacities can still differ because their workers and available hours are line-specific.


In [ ]:
pol_matrix_raw.columns = [
    clean_identifier(column)
    for column in pol_matrix_raw.columns
]
require_columns(pol_matrix_raw, ["Product_List"], "POL_Matrix")

pol_matrix_raw = pol_matrix_raw.loc[
    pol_matrix_raw["Product_List"].notna()
].copy()
pol_matrix_raw["Product_List"] = pol_matrix_raw["Product_List"].map(
    clean_identifier
)
require_unique(pol_matrix_raw["Product_List"], "products in POL_Matrix")

pol_products = set(pol_matrix_raw["Product_List"])
if pol_products != set(products):
    raise ValueError(
        "POL_Matrix products do not match Master_List. "
        f"Unknown={sorted(pol_products - set(products))}, "
        f"missing={sorted(set(products) - pol_products)}"
    )

pol_matrix = pol_matrix_raw.set_index("Product_List").reindex(products)
pol_matrix = pol_matrix.apply(pd.to_numeric, errors="raise")

if pol_matrix.isna().any().any():
    raise ValueError("POL_Matrix contains blank labor-time cells.")

if (pol_matrix < 0).any().any():
    raise ValueError("POL_Matrix labor times cannot be negative.")

fractional_pol = (pol_matrix - pol_matrix.round()).abs() > 1e-6
if fractional_pol.any().any():
    raise ValueError("POL_Matrix labor times must be whole milliseconds.")

pol_matrix = pol_matrix.round().astype("int64")

registered_processes = list(dict.fromkeys(line_process.values()))

unmapped_processes = [
    process
    for process in pol_matrix.columns
    if (pol_matrix[process] > 0).any()
    and process not in registered_processes
]

if unmapped_processes:
    raise ValueError(
        "POL_Matrix contains required processes with no registered line: "
        f"{unmapped_processes}"
    )

process_order = [
    process
    for process in pol_matrix.columns
    if process in registered_processes
]

product_routes = {
    product: [
        process
        for process in process_order
        if pol_matrix.loc[product, process] > 0
    ]
    for product in products
}

products_without_routes = [
    product
    for product, route in product_routes.items()
    if not route
]

if products_without_routes:
    raise ValueError(
        "Every product needs at least one positive POL process. "
        f"Missing routes: {products_without_routes}"
    )

process_lines = {
    process: [
        line
        for line in lines
        if line_process[line] == process
    ]
    for process in process_order
}

labor_time_per_unit_ms = {
    (product, process): int(pol_matrix.loc[product, process])
    for product in products
    for process in product_routes[product]
}

## 4. Load regular labor and the planning calendar

`Max_Hr` and `Work_Total` stay in hours in Excel. Python converts them to milliseconds at the input boundary. Model capacities, POL use, utilization thresholds, and salary calculations remain in milliseconds; reports convert them back to hours only for display.


In [ ]:
with pd.ExcelFile(LABOR_INPUT, engine="openpyxl") as workbook:
    required_labor_sheets = {"Expanded_Details", "Work_Total"}
    missing_labor_sheets = sorted(
        required_labor_sheets - set(workbook.sheet_names)
    )
    if missing_labor_sheets:
        raise ValueError(
            f"{LABOR_INPUT.name} is missing sheets: {missing_labor_sheets}"
        )

    shift_details = workbook.parse("Expanded_Details")
    work_total_raw = workbook.parse(
        "Work_Total",
        header=WORK_TOTAL_HEADER_ROW,
    )

shift_columns = [
    "Production_Line",
    "Shift",
    "Workers",
    "Max_Hr",
    "Hrly_Sal",
]
require_columns(shift_details, shift_columns, "Expanded_Details")

shift_details = shift_details.loc[
    shift_details["Production_Line"].notna()
    & shift_details["Shift"].notna(),
    shift_columns,
].copy()

shift_details["Production_Line"] = shift_details["Production_Line"].map(
    clean_identifier
)
shift_details["Shift"] = shift_details["Shift"].map(clean_identifier)

unknown_labor_lines = sorted(
    set(shift_details["Production_Line"]) - set(lines)
)
if unknown_labor_lines:
    raise ValueError(
        f"Expanded_Details contains unregistered lines: {unknown_labor_lines}"
    )

duplicate_line_shifts = shift_details.duplicated(
    subset=["Production_Line", "Shift"],
    keep=False,
)
if duplicate_line_shifts.any():
    duplicates = shift_details.loc[
        duplicate_line_shifts,
        ["Production_Line", "Shift"],
    ]
    raise ValueError(
        "Expanded_Details has duplicate line-shift rows:\n"
        f"{duplicates.to_string(index=False)}"
    )

shift_details["Workers"] = as_nonnegative_integers(
    shift_details["Workers"],
    "Expanded_Details.Workers",
)
shift_details[["Max_Hr", "Hrly_Sal"]] = shift_details[
    ["Max_Hr", "Hrly_Sal"]
].apply(pd.to_numeric, errors="raise")

if (shift_details["Max_Hr"] < 0).any():
    raise ValueError("Expanded_Details.Max_Hr cannot be negative.")

if (shift_details["Hrly_Sal"] < 0).any():
    raise ValueError("Expanded_Details.Hrly_Sal cannot be negative.")

active_shift_details = shift_details.loc[
    shift_details["Workers"] > 0
].copy()

daily_salary = active_shift_details.set_index(
    ["Production_Line", "Shift"]
)["Hrly_Sal"].to_dict()

invalid_active_hours = active_shift_details["Max_Hr"] <= 0
if invalid_active_hours.any():
    bad_rows = active_shift_details.loc[
        invalid_active_hours,
        ["Production_Line", "Shift", "Max_Hr"],
    ]
    raise ValueError(
        "An active line-shift must have positive Max_Hr:\n"
        f"{bad_rows.to_string(index=False)}"
    )

# Convert the Excel hour field once, then remove it from internal data.
active_shift_details = (
    active_shift_details.assign(
        Max_Time_MS=lambda df: (
            df["Max_Hr"] * MS_PER_HOUR
        ).round().astype("int64"),
        Salary_Per_MS=lambda df: (
            df["Hrly_Sal"] / df["Max_Time_MS"]
        ),
    )
    .drop(columns="Max_Hr")
)

active_line_shifts = list(
    active_shift_details[["Production_Line", "Shift"]].itertuples(
        index=False,
        name=None,
    )
)
line_workers = active_shift_details.set_index(["Production_Line", "Shift"])["Workers"].to_dict()
line_shift_max_time_ms = active_shift_details.set_index(["Production_Line", "Shift"])["Max_Time_MS"].to_dict()
salary_per_ms = active_shift_details.set_index(["Production_Line", "Shift"])["Salary_Per_MS"].to_dict()
Hrly_Salary = active_shift_details.set_index(["Production_Line", "Shift"])["Hrly_Sal"].to_dict()

shifts = clean_identifier_list(shift_details["Shift"].drop_duplicates())
active_lines = [
    line
    for line in lines
    if any(active_line == line for active_line, shift in active_line_shifts)
]
spare_lines = [line for line in lines if line not in active_lines]

shifts_by_line = {
    line: [
        shift
        for active_line, shift in active_line_shifts
        if active_line == line
    ]
    for line in active_lines
}


In [ ]:
# Work_Total has nine descriptive columns before its dated schedule.
if len(work_total_raw.columns) <= WORK_TOTAL_DATE_START_COLUMN:
    raise ValueError("Work_Total does not contain any dated availability columns.")

calendar_dates = [
    pd.Timestamp(pd.to_datetime(column, errors="raise")).normalize()
    for column in work_total_raw.columns[WORK_TOTAL_DATE_START_COLUMN:]
]
require_unique(calendar_dates, "dates in Work_Total")

if calendar_dates != sorted(calendar_dates):
    raise ValueError("Work_Total date columns must be in chronological order.")

days = list(range(1, len(calendar_dates) + 1))
day_to_date = dict(zip(days, calendar_dates))
date_to_day = dict(zip(calendar_dates, days))

work_total = work_total_raw.iloc[
    :,
    [
        0,
        1,
        2,
        *range(WORK_TOTAL_DATE_START_COLUMN, len(work_total_raw.columns)),
    ],
].copy()
work_total.columns = [
    "Line",
    "Shift",
    "Workers",
    *calendar_dates,
]

# Excel often shows the line name once for several shift rows.
work_total["Line"] = work_total["Line"].ffill()
work_total = work_total.loc[
    work_total["Shift"].notna()
    & work_total["Line"].ne("Grand Total")
].copy()
work_total["Line"] = work_total["Line"].map(clean_identifier)
work_total["Shift"] = work_total["Shift"].map(clean_identifier)

# Convert Excel hours to ms directly in the long table.
work_long = (
    work_total.melt(
        id_vars=["Line", "Shift"],
        value_vars=calendar_dates,
        var_name="Date",
        value_name="Regular_Time_MS",
    )
    .assign(
        Day=lambda df: df["Date"].map(date_to_day),
        Regular_Time_MS=lambda df: (
            pd.to_numeric(df["Regular_Time_MS"], errors="raise")
            .fillna(0.0)
            .mul(MS_PER_HOUR)
            .round()
            .astype("int64")
        ),
    )
)

if (work_long["Regular_Time_MS"] < 0).any():
    raise ValueError("Work_Total cannot contain negative available time.")

duplicate_availability = work_long.duplicated(
    subset=["Line", "Shift", "Day"],
    keep=False,
)
if duplicate_availability.any():
    duplicate_keys = work_long.loc[
        duplicate_availability,
        ["Line", "Shift", "Day"],
    ].drop_duplicates()
    raise ValueError(
        "Work_Total contains duplicate line-shift-day availability:\n"
        f"{duplicate_keys.to_string(index=False)}"
    )

excel_regular_time_ms = work_long.set_index(
    ["Line", "Shift", "Day"]
)["Regular_Time_MS"].to_dict()

regular_available_time_ms = {}
regular_labor_capacity_ms = {}
missing_availability = []

for line, shift in active_line_shifts:
    for day in days:
        key = (line, shift, day)

        if key not in excel_regular_time_ms:
            missing_availability.append(key)

        regular_available_time_ms[key] = min(
            int(excel_regular_time_ms.get(key, 0)),
            int(line_shift_max_time_ms[line, shift]),
        )
        regular_labor_capacity_ms[key] = (
            regular_available_time_ms[key]
            * int(line_workers[line, shift])
        )

if missing_availability:
    missing_line_shifts = sorted({
        (line, shift)
        for line, shift, day in missing_availability
    })
    print(
        "Work_Total is missing these active line-shifts; "
        "their missing dates are treated as zero availability:"
    )
    display(pd.DataFrame(missing_line_shifts, columns=["Line", "Shift"]))

# ceil() makes the 70% boundary exact because used POL time is integer ms.
regular_min_labor_ms = {
    key: math.ceil(labor_ms * MIN_TIME_UTILIZATION)
    for key, labor_ms in regular_labor_capacity_ms.items()
}


## 5. Previous-session WIP snapshot → opening WIP

`Last_MIP` stores one end-of-session state, not a calendar of movements:

| As_Of_Date | Product_List | Process | WIP_Units |
|---|---|---|---:|
| 2026-08-31 | A | Pri | 60 |
| 2026-08-31 | A | Cut | 25 |

Every row records a product-process balance. `As_Of_Date` must be the day before the new planning horizon starts. Physical lines never appear here: lines provide capacity, while processes own WIP state.

The sheet may include every process for convenience. Rows that are not WIP queues in the product's current route are left unused by the model; `WIP_Units` only needs to be a nonnegative integer.


In [ ]:
initial_wip = {
    (product, process): 0
    for product in products
    for process in product_routes[product][:-1]
}

if last_mip_raw.empty:
    if LAST_MIP_REQUIRED:
        raise ValueError(f"{LAST_MIP_SHEET} is empty.")
else:
    last_mip_columns = [
        "As_Of_Date",
        "Product_List",
        "Process",
        "WIP_Units",
    ]
    require_columns(last_mip_raw, last_mip_columns, LAST_MIP_SHEET)

    last_mip = (
        last_mip_raw
        .dropna(how="all")
        .loc[:, last_mip_columns]
        .assign(
            As_Of_Date=lambda df: pd.to_datetime(
                df["As_Of_Date"], errors="raise"
            ).dt.normalize(),
            Product_List=lambda df: df["Product_List"].map(clean_identifier),
            Process=lambda df: df["Process"].map(clean_identifier),
            WIP_Units=lambda df: as_nonnegative_integers(
                df["WIP_Units"],
                f"{LAST_MIP_SHEET}.WIP_Units",
            ),
        )
    )

    if last_mip["As_Of_Date"].isna().any():
        raise ValueError(f"{LAST_MIP_SHEET}.As_Of_Date contains blanks.")
    if last_mip["As_Of_Date"].nunique() != 1:
        raise ValueError(f"{LAST_MIP_SHEET} must contain one As_Of_Date.")

    snapshot_date = last_mip["As_Of_Date"].iloc[0]
    if snapshot_date != (
        calendar_dates[0] - pd.Timedelta(1, unit="D")
    ):
        raise ValueError(
            f"{LAST_MIP_SHEET} is dated {snapshot_date.date()}, "
            f"but the current period starts on {calendar_dates[0].date()}."
        )

    last_mip = last_mip.set_index(["Product_List", "Process"])
    require_unique(
        list(last_mip.index),
        f"product-process rows in {LAST_MIP_SHEET}",
    )

    initial_wip = {
        key: int(last_mip["WIP_Units"].get(key, 0))
        for key in initial_wip
    }

opening_wip_table = (
    pd.Series(initial_wip, name="Opening WIP Units")
    .rename_axis(["Product", "Completed Process"])
    .to_frame()
)

print("OPENING WIP FROM PREVIOUS-SESSION SNAPSHOT")
display(opening_wip_table)


# Optimization model

The model is built in five families:

1. feasible index sets;
2. decision variables;
3. reusable flow expressions;
4. constraints;
5. objective and solve.


## 6. Feasible index sets

Spare lines remain in master data and diagnostics, but a line with no positive-worker shift does not receive production variables.


In [ ]:
# Map each PROCESS to its currently staffed physical LINES.
#
# Example:
# process_lines["Pri"] = ["Pri1", "Pri2", "Pri3"]
# active_lines = ["Pri1", "Pri3", "Cut1"]
# active_process_lines["Pri"] -> ["Pri1", "Pri3"]
active_process_lines = {
    process: [
        line
        for line in process_lines[process]
        if line in active_lines
    ]
    for process in process_order
}


# Find processes that:
#   1. appear in at least one product route, and
#   2. currently have no staffed physical line.
#
# Example:
# If product A requires "Sew" but active_process_lines["Sew"] == [],
# this list will contain "Sew".
unavailable_required_processes = [
    process
    for process in process_order
    if any(process in product_routes[product] for product in products)
    and not active_process_lines[process]
]

if unavailable_required_processes:
    print(
        "Warning: these required processes have no currently staffed line: "
        f"{unavailable_required_processes}"
    )


# Valid combinations of PRODUCT and physical LINE.
#
# A product can use a line when the line's parent PROCESS belongs
# to that product's route.
#
# Example:
# line_process["Pri1"] = "Pri"
# product_routes["A"] = ["Pri", "Cut", "Sew", "Pac"]
# Therefore: ("A", "Pri1") is included.
#
# Notice that this is indexed by line because production capacity is
# line-specific, even though route eligibility is process-specific.
feasible_product_line_pairs = [
    (product, line)
    for product in products
    for line in active_lines
    if line_process[line] in product_routes[product]
]


# Reverse lookup showing which products can run on each physical line.
#
# Example:
# feasible_product_line_pairs contains:
# [("A", "Pri1"), ("B", "Pri1"), ("A", "Cut1")]
#
# products_by_line becomes:
# {
#     "Pri1": ["A", "B"],
#     "Cut1": ["A"],
# }
products_by_line = {
    line: [
        product
        for product, feasible_line in feasible_product_line_pairs
        if feasible_line == line
    ]
    for line in active_lines
}


# Keys for regular-production quantity variables.
#
# Index meaning:
# (product, physical line, shift, planning day)
#
# Example:
# ("A", "Pri1", "Day", 1)
# means product A produced on Pri1, during Day shift, on model day 1.
regular_production_keys = [
    (product, line, shift, day)
    for product, line in feasible_product_line_pairs
    for shift in shifts_by_line[line]
    for day in days
]


# Product-line-day keys without the shift dimension.
#
# These are used when regular production from all shifts and OT
# production need to be combined for the same line and day.
#
# Example:
# ("A", "Pri1", 1)
line_product_day_keys = [
    (product, line, day)
    for product, line in feasible_product_line_pairs
    for day in days
]


# Every active physical line on every planning day.
#
# There is no product or shift dimension here.
#
# Example:
# ("Pri1", 1), ("Pri1", 2), ("Cut1", 1), ...
line_day_keys = [
    (line, day)
    for line in active_lines
    for day in days
]


# Every active physical line-shift-day combination.
#
# These keys are used for regular-time capacity, utilization,
# and under-minimum-time calculations.
#
# Example:
# ("Pri1", "Day", 1)
# ("Pri1", "Rotate_A", 1)
line_shift_day_keys = [
    (line, shift, day)
    for line, shift in active_line_shifts
    for day in days
]


# Only open line-shift-days receive the 70% utilization classification.
#
# If Work_Total gives zero available worker-time, the shift is closed.
# It therefore receives neither production nor an idle-time penalty.
#
# Example:
# regular_labor_capacity_ms["Pri1", "Day", 1] > 0
#     -> ("Pri1", "Day", 1) is included.
#
# regular_labor_capacity_ms["Pri1", "Day", 2] == 0
#     -> ("Pri1", "Day", 2) is excluded.
under_min_time_keys = [
    (line, shift, day)
    for line, shift, day in line_shift_day_keys
    if regular_labor_capacity_ms[line, shift, day] > 0
]


# Keys for process-level WIP balances.
#
# Index meaning:
# (product, completed process, planning day)
#
# The final route process is excluded with [:-1] because output from
# that process is finished product, not work in process.
#
# Example:
# product_routes["A"] = ["Pri", "Cut", "Sew", "Pac"]
#
# WIP keys are created for:
# ("A", "Pri", day)
# ("A", "Cut", day)
# ("A", "Sew", day)
#
# No ("A", "Pac", day) key is created because Pac completes the product.
wip_keys = [
    (product, process, day)
    for product in products
    for process in product_routes[product][:-1]
    for day in days
]


# Safe upper bound for the total amount of each product that may move
# through its route.
#
# It includes:
#   new forecast quantity
#   + WIP already carried from every unfinished process
#
# Example:
# forecast["A"] = 100
# initial WIP: Pri=10, Cut=5, Sew=0
# max_route_quantity["A"] = 100 + 10 + 5 + 0 = 115
#
# This is only an activation/big-M upper bound. It is not the amount
# the model is required to produce.
max_route_quantity = {
    product: forecast[product] + sum(
        initial_wip[product, process]
        for process in product_routes[product][:-1]
    )
    for product in products
}


minimum_unscheduled_batch = {
    product: max(
        min(
            minimum_input_by_line[line]
            for line in active_process_lines[process]
        )
        for process in product_routes[product]
    )
    for product in products
}

## 7. Decision variables

| Variable | Index | Meaning |
|---|---|---|
| `production` | product, day | finished units leaving the final process |
| `regular_qty` | product, line, shift, day | units processed in regular time |
| `batch_active` | product, line, day | 1 when that product uses that line |
| `line_under_min_time` | line, shift, day | 1 when an open line-shift uses less than 70% of available worker-time |
| `under_min_time_salary_cost` | line, shift, day | salary value of unused labor on an under-minimum-time shift |
| `wip` | product, completed process, day | closing queue after that process |


In [ ]:
model = pl.LpProblem(
    "Regular_Production_Scheduling",
    pl.LpMaximize,
)

production = pl.LpVariable.dicts(
    "Production",
    [(product, day) for product in products for day in days],
    lowBound=0,
    cat="Integer",
)

regular_qty = pl.LpVariable.dicts(
    "RegularQty",
    regular_production_keys,
    lowBound=0,
    cat="Integer",
)

batch_active = pl.LpVariable.dicts(
    "BatchActive",
    line_product_day_keys,
    cat="Binary",
)

line_under_min_time = pl.LpVariable.dicts(
    "LineUnderMinTime",
    under_min_time_keys,
    cat="Binary",
)

under_min_time_salary_cost = pl.LpVariable.dicts(
    "UnderMinTimeSalaryCost",
    under_min_time_keys,
    lowBound=0,
    cat="Continuous",
)

wip = pl.LpVariable.dicts(
    "WIP",
    wip_keys,
    lowBound=0,
    cat="Integer",
)

has_unscheduled = pl.LpVariable.dicts(
    "HasUnscheduled",
    products,
    cat="Binary",
)

## 8. Reusable production-flow expressions

Quantity decisions remain line-specific because each physical line has its own capacity. Route flow is then summed to the parent process.

Regular labor uses `labor_time_per_unit_ms[product, line_process[line]]`: the line chooses where production occurs, while `line_process[line]` selects the correct process-level POL standard.


In [ ]:
# 1) Regular quantity processed by one physical line on one day.
# Sum the line's separate shift decisions before route-flow calculations.
line_product_day_processed = {
    (product, line, day): pl.lpSum(
        regular_qty[product, line, shift, day]
        for shift in shifts_by_line[line]
    )
    for product, line, day in line_product_day_keys
}


# 2) Product quantity processed by one route process on one day.
# Multiple physical lines belonging to the same process are added here.
stage_processed = {
    (product, process, day): pl.lpSum(
        line_product_day_processed[product, line, day]
        for line in active_process_lines[process]
        if (product, line, day) in line_product_day_processed
    )
    for product in products
    for process in product_routes[product]
    for day in days
}


# 3) Total product units processed by one line-shift on one day.
line_shift_day_processed = {
    (line, shift, day): pl.lpSum(
        regular_qty[product, line, shift, day]
        for product in products_by_line[line]
    )
    for line, shift, day in line_shift_day_keys
}


# 4) Total regular units processed by one physical line on one day.
line_day_processed = {
    (line, day): pl.lpSum(
        line_product_day_processed[product, line, day]
        for product in products_by_line[line]
    )
    for line, day in line_day_keys
}


# 5) Person-time used by one line-shift on one regular day.
regular_labor_used_ms = {
    (line, shift, day): pl.lpSum(
        labor_time_per_unit_ms[product, line_process[line]]
        * regular_qty[product, line, shift, day]
        for product in products_by_line[line]
    )
    for line, shift, day in line_shift_day_keys
}


# 6) Unscheduled quantity of each product after all days
unscheduled_qty = {
    product: (
        forecast[product]
        - pl.lpSum(production[product, day] for day in days)
    )
    for product in products
}

## 9. Forecast, ordered route, and WIP balance

Finished output stays below forecast. For adjacent route processes (r \rightarrow r+1):

\[
WIP_{p,r,d}=OpeningWIP_{p,r,d}+Processed_{p,r,d}-Processed_{p,r+1,d}
\]

Because WIP is nonnegative, downstream cumulative output cannot exceed available upstream output.


In [ ]:
for product in products:
    route = product_routes[product]
    final_process = route[-1]

    model += (
        pl.lpSum(production[product, day] for day in days)
        <= forecast[product],
        f"Forecast_Limit_{product}",
    )

    # Ensure that any unscheduled quantity is at least the minimum batch size
    model += (
        unscheduled_qty[product]
        >= max(
            min(
                minimum_input_by_line[line]
                for line in active_process_lines[process]
            )
            for process in product_routes[product]
        ) * has_unscheduled[product],
        f"Unscheduled_Minimum_Batch_{product}",
    )

    model += (
        unscheduled_qty[product]
        <= forecast[product] * has_unscheduled[product],
        f"Unscheduled_Activation_{product}",
    )

    for day in days:
        model += (
            production[product, day]
            == stage_processed[product, final_process, day],
            f"Finished_Output_{product}_{day}",
        )

    for current_process, next_process in zip(route, route[1:]):
        for day_position, day in enumerate(days):
            if day_position == 0:
                opening_units = initial_wip[product, current_process]
            else:
                previous_day = days[day_position - 1]
                opening_units = wip[product, current_process, previous_day]

            model += (
                wip[product, current_process, day]
                == opening_units
                + stage_processed[product, current_process, day]
                - stage_processed[product, next_process, day],
                f"WIP_Balance_{product}_{current_process}_{day}",
            )

            if not ALLOW_SAME_DAY_TRANSFER:
                model += (
                    stage_processed[product, next_process, day]
                    <= opening_units,
                    f"Prior_Day_Release_{product}_{next_process}_{day}",
                )

        if CLEAR_ENDING_WIP:
            model += (
                wip[product, current_process, days[-1]] == 0,
                f"Clear_Ending_WIP_{product}_{current_process}",
            )


## 10. Minimum batches and time-based KPI policy

`Minimum_Input` is a unit-based batch rule for a product scheduled on a physical line. The old unit-based `Max_KPI` and `Min_KPI` constraints are not used.

For a regular line-shift-day, each line uses the POL of its parent process:

\[
UsedTime_{l,s,d}=\sum_p POL_{p,\,process(l)}\times RegularQty_{p,l,s,d}
\]

The maximum time is the dated worker-time supplied by `Work_Total`. The minimum-utilization threshold is 70% of that capacity. Falling below 70% is allowed but incurs the salary value of all unused available worker-time; at or above 70%, that special penalty is zero.


In [ ]:
# ---------- Minimum product input on a physical line ----------
for product, line, day in line_product_day_keys:
    quantity = line_product_day_processed[product, line, day]
    active = batch_active[product, line, day]

    model += (
        quantity >= minimum_input_by_line[line] * active,
        f"Minimum_Input_{product}_{line}_{day}",
    )
    model += (
        quantity <= max_route_quantity[product] * active,
        f"Batch_Activation_{product}_{line}_{day}",
    )

## 11. Regular time capacity and 70% classification

Every staffed physical line-shift has its own dated capacity:

\[
\sum_p POL_{p,\,process(l)}\times RegularQty_{p,l,s,d}
\le Workers_{l,s}\times AvailableTime_{l,s,d}
\]

The same person-time expression is compared with 70% of that exact capacity. POL remains process-specific even though the capacity constraint is line-specific.


In [ ]:
for line, shift in active_line_shifts:
    for day in days:
        key = (line, shift, day)
        used_regular_labor_ms = regular_labor_used_ms[key]
        available_regular_labor_ms = regular_labor_capacity_ms[key]

        # Maximum time KPI: never exceed the dated Work_Total capacity.
        model += (
            used_regular_labor_ms <= available_regular_labor_ms,
            f"Regular_Time_Max_{line}_{shift}_{day}",
        )

        if key not in line_under_min_time:
            continue

        under_min_time = line_under_min_time[key]
        minimum_regular_labor_ms = regular_min_labor_ms[key]

        # Exact classification:
        #   UnderMinTime=0 -> used time is at least 70% of capacity.
        #   UnderMinTime=1 -> used time is strictly below that threshold.
        model += (
            used_regular_labor_ms
            >= minimum_regular_labor_ms * (1 - under_min_time),
            f"Regular_Time_Min_Class_Lower_{line}_{shift}_{day}",
        )
        model += (
            used_regular_labor_ms
            <= (minimum_regular_labor_ms - 1) * under_min_time
            + available_regular_labor_ms * (1 - under_min_time),
            f"Regular_Time_Min_Class_Upper_{line}_{shift}_{day}",
        )

## 12. Under-minimum-time salary cost and objective

`Daily_Sal` is salary per person for the full `Max_Time_MS`, so the model stores `Salary_Per_MS = Daily_Sal / Max_Time_MS`.

If regular POL labor is below 70% of available worker-time, the auxiliary cost variable is forced to equal the salary value of all unused available worker-time. At or above 70%, it is forced to zero. The objective maximizes regular finished-product profit minus this under-minimum-time cost.


In [ ]:
# ---------- Salary value of unused time while below 70% ----------
for line, shift, day in under_min_time_keys:
    key = (line, shift, day)
    under_min_time = line_under_min_time[key]
    cost = under_min_time_salary_cost[key]

    maximum_cost = (
        salary_per_ms[line, shift]
        * regular_labor_capacity_ms[key]
    )
    unused_salary = (
        salary_per_ms[line, shift]
        * (
            regular_labor_capacity_ms[key]
            - regular_labor_used_ms[key]
        )
    )

    # Exact linearization of:
    # cost = under_min_time * unused_salary
    model += (
        cost >= unused_salary - maximum_cost * (1 - under_min_time),
        f"Under_Min_Time_Cost_Lower_{line}_{shift}_{day}",
    )
    model += (
        cost <= unused_salary + maximum_cost * (1 - under_min_time),
        f"Under_Min_Time_Cost_Upper_{line}_{shift}_{day}",
    )
    model += (
        cost <= maximum_cost * under_min_time,
        f"Under_Min_Time_Cost_Switch_{line}_{shift}_{day}",
    )


total_production_profit = pl.lpSum(
    profit_per_product[product] * production[product, day]
    for product in products
    for day in days
)

total_under_min_time_salary_cost = pl.lpSum(
    under_min_time_salary_cost[line, shift, day]
    for line, shift, day in under_min_time_keys
)

model += (
    total_production_profit
    - total_under_min_time_salary_cost,
    "Maximize_Regular_Net_Profit",
)


## 13. Model checkpoint and solve


In [ ]:
num_binary = sum(variable.isBinary() for variable in model.variables())
num_integer = sum(
    variable.cat == pl.LpInteger and not variable.isBinary()
    for variable in model.variables()
)
num_continuous = sum(
    variable.cat == pl.LpContinuous
    for variable in model.variables()
)

print(f"Total Variables:   {len(model.variables()):,}")
print(f"  • Continuous:    {num_continuous:,}")
print(f"  • General Int:   {num_integer:,}")
print(f"  • Binary (0/1):  {num_binary:,}")
print(f"Total Constraints: {len(model.constraints):,}")

solver = pl.HiGHS(
    gapRel=SOLVER_RELATIVE_GAP,
    timeLimit=SOLVER_TIME_LIMIT_SECONDS,
    msg=SOLVER_LOG,
)
model.solve(solver)

solve_status = pl.LpStatus[model.status]
print("Status:", solve_status)

if solve_status not in {"Optimal", "Feasible"}:
    raise RuntimeError(
        f"The model has no reportable solution. Solver status: {solve_status}"
    )


In [ ]:
def solution_value(item):
    """Return a solved PuLP value as a regular Python float."""
    value = pl.value(item)
    return 0.0 if value is None else float(value)


objective_summary = pd.Series({
    "Finished Production Profit": solution_value(total_production_profit),
    "Under-70%-Time Idle Salary Cost": solution_value(
        total_under_min_time_salary_cost
    ),
    "Net Objective": solution_value(model.objective),
})

print("OBJECTIVE SUMMARY")
display(objective_summary.to_frame("Value").style.format("{:,.2f}"))

production_result = export_utils.export_daily_production_schedule(
    model=model,
    production=production,
    products=products,
    days=days,
    day_to_date=day_to_date,
    planning_input=PLANNING_INPUT,
)

print("FINISHED-PRODUCT DAILY SCHEDULE")
display(production_result)


# Reports

Reports are separate from model construction. Changing a table below cannot change the optimization result above.


In [ ]:
def make_product_day_pivot(
    rows,
    group_columns,
    product_order,
    date_order,
):
    """Put products on rows and planning dates on columns.

    The regular schedule uses Line / Shift / Product as its row index.
    ``date_order`` keeps zero-output planning days visible.
    """
    index_columns = [*group_columns, "Product"]

    if not rows:
        empty_index = pd.MultiIndex.from_arrays(
            [[] for column in index_columns],
            names=index_columns,
        )
        return pd.DataFrame(
            index=empty_index,
            columns=[*date_order, "Total"],
        ).fillna(0)

    row_frame = pd.DataFrame(rows)
    row_frame["Product"] = pd.Categorical(
        row_frame["Product"],
        categories=product_order,
        ordered=True,
    )
    table = row_frame.pivot_table(
        index=index_columns,
        columns="Date",
        values="Quantity",
        aggfunc="sum",
        fill_value=0,
        sort=False,
        observed=True,
    ).reindex(columns=date_order, fill_value=0)
    table = table.round().astype(int)
    table["Total"] = table.sum(axis=1)
    table.columns.name = None
    return table


## 14. Forecast attainment


In [ ]:
forecast_attainment_rows = []

for product in products:
    finished_units = sum(
        solution_value(production[product, day])
        for day in days
    )
    product_forecast_units = forecast[product]

    forecast_attainment_rows.append({
        "Product": product,
        "Forecast": product_forecast_units,
        "Finished": finished_units,
        "Unscheduled": max(0.0, product_forecast_units - finished_units),
        "Attainment": safe_ratio(finished_units, product_forecast_units),
        "Profit per Product": profit_per_product[product],
    })

forecast_attainment_table = pd.DataFrame(
    forecast_attainment_rows
).set_index("Product")

print("FORECAST ATTAINMENT")
display(
    forecast_attainment_table.style.format({
        "Forecast": "{:,.0f}",
        "Finished": "{:,.0f}",
        "Unscheduled": "{:,.0f}",
        "Attainment": "{:.1%}",
        "Profit per Product": "{:,.2f}",
    })
)


## 15. Regular production schedule and daily line output

The regular schedule keeps physical lines and shifts on rows, with planning dates on columns. The daily line table then sums all regular shifts for each physical line.


In [ ]:
regular_rows = []

for day in days:
    schedule_date = pd.Timestamp(day_to_date[day]).date()

    for line, shift in active_line_shifts:
        for product in products_by_line[line]:
            quantity = solution_value(
                regular_qty[product, line, shift, day]
            )

            if quantity > SOLUTION_TOLERANCE:
                regular_rows.append({
                    "Date": schedule_date,
                    "Line": line,
                    "Shift": shift,
                    "Product": product,
                    "Quantity": quantity,
                })

schedule_dates = [
    pd.Timestamp(day_to_date[day]).date()
    for day in days
]

regular_schedule_table = make_product_day_pivot(
    regular_rows,
    ["Line", "Shift"],
    products,
    schedule_dates,
)

if regular_rows:
    print("REGULAR PRODUCTION SCHEDULE")
    display(
        display_utils.style_grouped_table(
            regular_schedule_table,
            {
                column: "{:,.0f}"
                for column in regular_schedule_table.columns
            },
        )
    )
else:
    print("No regular production was scheduled.")


In [ ]:
daily_line_output_rows = []

for line, day in line_day_keys:
    total_units = solution_value(line_day_processed[line, day])

    if total_units > SOLUTION_TOLERANCE:
        daily_line_output_rows.append({
            "Date": pd.Timestamp(day_to_date[day]).date(),
            "Line": line,
            "Regular Units": total_units,
        })

if daily_line_output_rows:
    daily_line_output_table = pd.DataFrame(
        daily_line_output_rows
    ).set_index(["Date", "Line"])
    print("DAILY REGULAR LINE OUTPUT")
    display(
        daily_line_output_table.style.format({
            "Regular Units": "{:,.0f}",
        })
    )
else:
    daily_line_output_table = pd.DataFrame()
    print("No line output was scheduled.")


## 16. Remaining regular time and under-minimum-time cost


In [ ]:
remaining_time_rows = []
under_min_time_rows = []

for day in days:
    schedule_date = pd.Timestamp(day_to_date[day]).date()

    for line, shift in active_line_shifts:
        key = (line, shift, day)
        available_labor_ms = regular_labor_capacity_ms[key]
        used_labor_ms = solution_value(regular_labor_used_ms[key])
        workers = int(line_workers[line, shift])

        # Convert ms to hours only in values written to report rows.
        remaining_time_rows.append({
            "Date": schedule_date,
            "Line": line,
            "Shift": shift,
            "Remaining Line Hours": (
                max(0.0, available_labor_ms - used_labor_ms)
                / workers
                / MS_PER_HOUR
            ),
        })

        if (
            key in line_under_min_time
            and solution_value(line_under_min_time[key]) > 0.5
        ):
            under_min_time_rows.append({
                "Date": schedule_date,
                "Line": line,
                "Shift": shift,
                "Regular Units": solution_value(
                    line_shift_day_processed[key]
                ),
                "Minimum Time Utilization": MIN_TIME_UTILIZATION,
                "Actual Time Utilization": safe_ratio(
                    used_labor_ms,
                    available_labor_ms,
                ),
                "Available Line Hours": (
                    regular_available_time_ms[key] / MS_PER_HOUR
                ),
                "Minimum Line Hours": (
                    regular_min_labor_ms[key]
                    / workers
                    / MS_PER_HOUR
                ),
                "Used Line Hours": (
                    used_labor_ms / workers / MS_PER_HOUR
                ),
                "Unused Line Hours": (
                    max(0.0, available_labor_ms - used_labor_ms)
                    / workers
                    / MS_PER_HOUR
                ),
                "Line Workers": workers,
                "Daily Salary per Person": daily_salary[line, shift],
                "Under-Min-Time Salary Cost": solution_value(
                    under_min_time_salary_cost[key]
                ),
                "Under-Min-Time Cost per Person": (
                    solution_value(under_min_time_salary_cost[key])
                    / workers
                ),
            })

# Pivot table with Line as row index and Date/Shift as multi-index columns
remaining_time_table = (
    pd.DataFrame(remaining_time_rows)
    .pivot_table(
        index="Line",
        columns=["Date", "Shift"],
        values="Remaining Line Hours",
        aggfunc="sum",
        fill_value=0.0,
        sort=False,
    )
)

dates = [pd.Timestamp(day_to_date[d]).date() for d in days]
multi_cols = pd.MultiIndex.from_product([dates, shifts], names=["Date", "Shift"])
remaining_time_table = remaining_time_table.reindex(columns=multi_cols, fill_value=0.0)

# 1. Calculate overall grand total before adding per-day totals
grand_total = remaining_time_table.sum(axis=1)

# 2. Add a 'Total' column for each individual date
for d in dates:
    remaining_time_table[(d, "Total")] = remaining_time_table[d].sum(axis=1)

# 3. Reorder columns so 'Total' sits at the end of each date's shifts
ordered_cols = []
for d in dates:
    for s in shifts:
        ordered_cols.append((d, s))
    ordered_cols.append((d, "Total"))

remaining_time_table = remaining_time_table[ordered_cols]

# 4. Append the overall Grand Total column at the far right
remaining_time_table[("Grand Total", "")] = grand_total

print("REGULAR WORKER-HOURS REMAINING")
display(
    display_utils.style_grouped_table(
        remaining_time_table,
        display_utils.remaining_time_format,
    )
)

if under_min_time_rows:
    under_min_time_cost_table = pd.DataFrame(
        under_min_time_rows
    ).set_index(["Date", "Line", "Shift"])
    print("UNDER-70%-TIME UNUSED-SALARY COST")
    display(
        under_min_time_cost_table.style.format({
            "Regular Units": "{:,.0f}",
            "Minimum Time Utilization": "{:.0%}",
            "Actual Time Utilization": "{:.1%}",
            "Available Line Hours": "{:,.2f}",
            "Minimum Line Hours": "{:,.2f}",
            "Used Line Hours": "{:,.2f}",
            "Unused Line Hours": "{:,.2f}",
            "Line Workers": "{:,.0f}",
            "Hourly Salary per Person": "{:,.2f}",
            "Under-Min-Time Salary Cost": "{:,.2f}",
            "Under-Min-Time Cost per Person": "{:,.2f}",
        })
    )
else:
    under_min_time_cost_table = pd.DataFrame()
    print("Every open line-shift reached the minimum time utilization.")


## 17. Solved line utilization

This table reports every active **line-shift** separately. Output units remain visible, but capacity and bottleneck signals are based entirely on POL worker-time. There is no separate hard unit KPI competing with product processing times.


In [ ]:
line_utilization_rows = []

for line, shift in active_line_shifts:
    regular_labor_available_ms = sum(
        regular_labor_capacity_ms[line, shift, day]
        for day in days
    )
    regular_labor_used_total_ms = sum(
        solution_value(regular_labor_used_ms[line, shift, day])
        for day in days
    )
    regular_units = sum(
        solution_value(line_shift_day_processed[line, shift, day])
        for day in days
    )
    open_days = [
        day
        for day in days
        if regular_labor_capacity_ms[line, shift, day] > 0
    ]
    under_min_time_days = sum(
        solution_value(line_under_min_time[line, shift, day]) > 0.5
        for day in open_days
        if (line, shift, day) in line_under_min_time
    )
    under_min_time_cost = sum(
        solution_value(under_min_time_salary_cost[line, shift, day])
        for day in open_days
        if (line, shift, day) in under_min_time_salary_cost
    )
    daily_time_loads = [
        safe_ratio(
            solution_value(regular_labor_used_ms[line, shift, day]),
            regular_labor_capacity_ms[line, shift, day],
        )
        for day in open_days
    ]
    full_time_days = sum(load >= 0.999 for load in daily_time_loads)

    if full_time_days:
        solved_signal = "Regular time binding"
    elif under_min_time_days:
        solved_signal = "Under minimum time"
    else:
        solved_signal = "Regular time remains"

    # Convert ms to hours only in values written to this display table.
    line_utilization_rows.append({
        "Process": line_process[line],
        "Line": line,
        "Shift": shift,
        "Minimum Time Utilization": MIN_TIME_UTILIZATION,
        "Regular Units": regular_units,
        "Regular Worker-Hours Used": (
            regular_labor_used_total_ms / MS_PER_HOUR
        ),
        "Regular Worker-Hours Available": (
            regular_labor_available_ms / MS_PER_HOUR
        ),
        "Period Regular Time Utilization": safe_ratio(
            regular_labor_used_total_ms,
            regular_labor_available_ms,
        ),
        "Peak Daily Regular Time Utilization": max(
            daily_time_loads,
            default=0.0,
        ),
        "Full Regular-Time Days": full_time_days,
        "Under-Min-Time Days": under_min_time_days,
        "Under-Min-Time Salary Cost": under_min_time_cost,
        "Solved Signal": solved_signal,
    })

line_utilization_table = pd.DataFrame(line_utilization_rows).set_index(
    ["Process", "Line", "Shift"]
)

print("SOLVED REGULAR LINE-SHIFT TIME UTILIZATION")
display(
    line_utilization_table.style.format({
        "Minimum Time Utilization": "{:.0%}",
        "Regular Units": "{:,.0f}",
        "Regular Worker-Hours Used": "{:,.2f}",
        "Regular Worker-Hours Available": "{:,.2f}",
        "Period Regular Time Utilization": "{:.1%}",
        "Peak Daily Regular Time Utilization": "{:.1%}",
        "Under-Min-Time Salary Cost": "{:,.2f}",
    })
)

if spare_lines:
    spare_line_table = line_config.loc[
        spare_lines,
        ["Minimum_Input"],
    ].copy()
    spare_line_table.insert(
        0,
        "Process",
        [line_process[line] for line in spare_line_table.index],
    )
    spare_line_table.index.name = "Spare Line"

    print("REGISTERED SPARE LINES WITH NO CURRENT WORKERS")
    display(spare_line_table)


## 18. Product routes, WIP carryover, and ending snapshot

The detailed table shows WIP movement through the current plan. `ending_wip_snapshot` then records only the exact end-of-horizon state using the same four-column structure expected by `Last_MIP` for the next session.

The snapshot remains process-level even when several physical lines serve the same process.


In [ ]:
product_route_table = pd.DataFrame(
    {
        "Route": [
            " → ".join(product_routes[product])
            for product in products
        ]
    },
    index=pd.Index(products, name="Product"),
)

print("PRODUCT ROUTES")
display(product_route_table)


In [ ]:
wip_rows = []

for day_position, day in enumerate(days):
    schedule_date = pd.Timestamp(day_to_date[day]).date()

    for product in products:
        route = product_routes[product]

        for current_process, next_process in zip(route, route[1:]):
            if day_position == 0:
                opening_units = float(initial_wip[product, current_process])
            else:
                previous_day = days[day_position - 1]
                opening_units = solution_value(
                    wip[product, current_process, previous_day]
                )

            added_today = solution_value(
                stage_processed[product, current_process, day]
            )
            moved_forward_today = solution_value(
                stage_processed[product, next_process, day]
            )
            closing_units = solution_value(
                wip[product, current_process, day]
            )

            if (
                opening_units <= SOLUTION_TOLERANCE
                and closing_units <= SOLUTION_TOLERANCE
            ):
                continue

            wip_rows.append({
                "Date": schedule_date,
                "Product": product,
                "Queue": f"{current_process} → {next_process}",
                "Opening Carryover Units": opening_units,
                "Added Today": added_today,
                "Moved Forward Today": -moved_forward_today,
                "Closing Carryover Units": closing_units,
            })

if wip_rows:
    wip_carryover_table = pd.DataFrame(wip_rows).set_index(
        ["Date", "Product", "Queue"]
    )
    print("WORK-IN-PROCESS CARRYOVER")
    display(
        display_utils.style_grouped_table(
            wip_carryover_table,
            display_utils.wip_display_format,
        )
    )
else:
    wip_carryover_table = pd.DataFrame()
    print("No work-in-process carryover was scheduled.")

last_day = days[-1]

ending_wip_snapshot = pd.DataFrame([
    {
        "As_Of_Date": pd.Timestamp(day_to_date[last_day]).date(),
        "Product_List": product,
        "Process": process,
        "WIP_Units": round(solution_value(wip[product, process, last_day])),
    }
    for product in products
    for process in product_routes[product][:-1]
])

ending_wip_table = ending_wip_snapshot.set_index(
    ["As_Of_Date", "Product_List", "Process"]
)

print("ENDING WIP SNAPSHOT FOR THE NEXT PLANNING SESSION")
display(ending_wip_table)


In [ ]:
# Sum finished regular production over the entire planning period.
solved_regular = {
    product: int(round(sum(
        solution_value(production[product, day])
        for day in days
    )))
    for product in products
}


workbook = load_workbook(PRODUCTION_INPUT, keep_vba=True)
sheet = workbook["Product_Forecast"]

headers = {
    clean_identifier(cell.value): cell.column
    for cell in sheet[1]
    if cell.value is not None
}

required_columns = [
    "Product_List",
    "Solved_Regular",
    "For_OT",
]

missing_columns = [
    column
    for column in required_columns
    if column not in headers
]

if missing_columns:
    raise ValueError(
        f"Product_Forecast is missing columns: {missing_columns}"
    )


for row in range(2, sheet.max_row + 1):
    raw_product = sheet.cell(
        row=row,
        column=headers["Product_List"],
    ).value

    if raw_product is None:
        continue

    product = clean_identifier(raw_product)

    if product not in solved_regular:
        continue

    sheet.cell(
        row=row,
        column=headers["Solved_Regular"],
        value=solved_regular[product],
    )

    sheet.cell(
        row=row,
        column=headers["For_OT"],
        value=max(
            0,
            int(forecast[product]) - solved_regular[product],
        ),
    )


workbook.save(PRODUCTION_INPUT)
workbook.close()

print("Solved_Regular and For_OT were written to Product_Forecast.")

In [ ]:
WRITE_MIP_SNAPSHOT = True

if WRITE_MIP_SNAPSHOT:
    snapshot_columns = [
        "As_Of_Date",
        "Product_List",
        "Process",
        "WIP_Units",
    ]

    mip_snapshot = ending_wip_snapshot.loc[
        :,
        snapshot_columns,
    ].copy()

    if (mip_snapshot["WIP_Units"] < 0).any():
        raise ValueError("Ending WIP cannot contain negative values.")

    workbook = load_workbook(
        PRODUCTION_INPUT,
        keep_vba=True,
    )

    if LAST_MIP_SHEET in workbook.sheetnames:
        sheet = workbook[LAST_MIP_SHEET]

        # Clear the previous snapshot while retaining the sheet itself.
        for row in sheet.iter_rows():
            for cell in row:
                cell.value = None
    else:
        sheet = workbook.create_sheet(LAST_MIP_SHEET)

    # Write headers.
    for column_number, column_name in enumerate(
        snapshot_columns,
        start=1,
    ):
        sheet.cell(
            row=1,
            column=column_number,
            value=column_name,
        )

    # Write the new process-level WIP snapshot.
    for row_number, values in enumerate(
        mip_snapshot.itertuples(index=False, name=None),
        start=2,
    ):
        for column_number, value in enumerate(values, start=1):
            sheet.cell(
                row=row_number,
                column=column_number,
                value=value,
            )

        sheet.cell(
            row=row_number,
            column=1,
        ).number_format = "yyyy-mm-dd"

    workbook.save(PRODUCTION_INPUT)
    workbook.close()

    print(
        f"Ending WIP snapshot written to "
        f"{PRODUCTION_INPUT.name}/{LAST_MIP_SHEET}."
    )
else:
    print("Last_MIP writeback is disabled.")